In [ ]:
from pathlib import Path
import subprocess
import os
import sys

repo = Path("/content/yolov5")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/yolov5.git", str(repo)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(repo / "requirements.txt"),
        "ultralytics==8.4.215", "roboflow",
    ],
    check=True,
)

# Avoid interactive experiment-tracker login prompts
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["COMET_MODE"] = "DISABLED"

from google.colab import drive
drive.mount("/content/drive")

import torch
import ultralytics

assert torch.cuda.is_available(), "Please connect a GPU first"
print("GPU: ", torch.cuda.get_device_name(0))
print("PyTorch: ", torch.__version__)
print("Ultralytics: ", ultralytics.__version__)

repo_commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True,
).strip()
print("YOLOv5 git commit: ", repo_commit)

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU:  NVIDIA L4
PyTorch:  2.14.1+cu130
Ultralytics:  8.4.215
YOLOv5 git commit:  33ae15ef99bd04ce1d3f12155b8a85f355d73351


In [ ]:
!pip install roboflow

from roboflow import Roboflow
import os
# Set ROBOFLOW_API_KEY in Colab Secrets or environment before running
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY"))
project = rf.workspace("sjp-gcfju").project("2026_greg")
version = project.version(4)
dataset = version.download("yolov5")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to 2026_GreG-4 in yolov5pytorch:: 100%|██████████| 3964/3964 [00:00<00:00, 8367.81it/s]


In [ ]:
import yaml

reference_run = Path(
    "/content/drive/MyDrive/paper_cup_yolo11/"
    "paper_cups_416_full120_20261119_114137"
)

reference_args = yaml.safe_load(
    (reference_run / "args.yaml").read_text(encoding="utf-8")
)

dataset_root = Path(dataset.location).resolve()

assert dataset_root.name == "2026_GreG-4", (
    f"Current directory is {dataset_root}, please confirm this is version 4 matching the previous YOLO11n run"
)

config = yaml.safe_load(
    (dataset_root / "data.yaml").read_text(encoding="utf-8")
)

names = config["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda k: int(k))]

expected_names = [
    "Back", "Front", "left_side",
    "right_side", "upright", "upside_down",
]
assert names == expected_names, f"Class order differs from previous run: {names}"

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
counts = {}

for split, folder in [("train", "train"), ("val", "valid"), ("test", "test")]:
    image_dir = dataset_root / folder / "images"
    label_dir = dataset_root / folder / "labels"

    assert image_dir.is_dir(), f"Missing image directory: {image_dir}"
    assert label_dir.is_dir(), f"Missing label directory: {label_dir}"

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    ]
    assert images, f"{folder} has no images"

    # Empty label files are allowed; they represent background-only images
    for label_file in label_dir.glob("*.txt"):
        for line in label_file.read_text(encoding="utf-8").splitlines():
            if line.strip():
                assert len(line.split()) == 5, (
                    f"Not in 5-column YOLO bounding-box format: {label_file}"
                )

    counts[split] = len(images)
    config[split] = str(image_dir)
    print(f"{folder}: {len(images)} images")

assert counts["val"] == 26 and counts["test"] == 22, (
    "Validation/test image counts differ from previous run; verify dataset version first"
)

config.update(path=str(dataset_root), names=names, nc=len(names))

data_yaml = dataset_root / "data_yolov5.yaml"
data_yaml.write_text(
    yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("Classes: ", names)
print("Training config: ", data_yaml)

train: 1932 images
valid: 26 images
test: 22 images
Classes:  ['Back', 'Front', 'left_side', 'right_side', 'upright', 'upside_down']
Training config:  /content/2026_GreG-4/data_yolov5.yaml


In [ ]:
hyp = yaml.safe_load(
    (repo / "data/hyps/hyp.scratch-low.yaml").read_text()
)

# Inherit matching hyperparameters from the previous YOLO11n configuration
shared_keys = [
    "lrf", "weight_decay", "warmup_epochs", "warmup_momentum",
    "hsv_h", "hsv_s", "hsv_v",
    "degrees", "translate", "scale", "shear", "perspective",
    "flipud", "fliplr", "mosaic", "mixup", "copy_paste",
]

for key in shared_keys:
    if key in reference_args:
        hyp[key] = reference_args[key]

hyp.update(
    lr0=0.001,
    momentum=0.9,
    warmup_bias_lr=0.0,
    degrees=0.0,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
)

hyp_yaml = Path("/content/hyp_v5s_compare11n.yaml")
hyp_yaml.write_text(
    yaml.safe_dump(hyp, sort_keys=False),
    encoding="utf-8",
)

print("Optimizer: AdamW")
print("Input 416, batch 8, 120 epochs, patience 0, seed 42")
print(yaml.safe_dump(hyp, sort_keys=False))

Optimizer: AdamW
Input 416, batch 8, 120 epochs, patience 0, seed 42
lr0: 0.001
lrf: 0.01
momentum: 0.9
weight_decay: 0.0005
warmup_epochs: 3.0
warmup_momentum: 0.8
warmup_bias_lr: 0.0
box: 0.05
cls: 0.5
cls_pw: 1.0
obj: 1.0
obj_pw: 1.0
iou_t: 0.2
anchor_t: 4.0
fl_gamma: 0.0
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.0
mosaic: 0.0
mixup: 0.0
copy_paste: 0.0



In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display
import time
import json

run_name = "v5s_416_full120_" + datetime.now(
    ZoneInfo("Europe/Stockholm")
).strftime("%Y%m%d_%H%M%S")

project_dir = Path("/content/drive/MyDrive/paper_cup_yolov5s")
project_dir.mkdir(parents=True, exist_ok=True)

run_dir = project_dir / run_name
log_path = project_dir / f"{run_name}.log"

command = [
    sys.executable, "-u", str(repo / "train.py"),
    "--weights", "yolov5s.pt",
    "--data", str(data_yaml),
    "--hyp", str(hyp_yaml),
    "--imgsz", "416",
    "--epochs", "120",
    "--batch-size", "8",
    "--patience", "0",
    "--optimizer", "AdamW",
    "--device", "0",
    "--workers", "2",
    "--seed", "42",
    "--project", str(project_dir),
    "--name", run_name,
]

if reference_args.get("cos_lr", False):
    command.append("--cos-lr")

# Record code version and environment for comparison
record = {
    "reference_yolo11_run": str(reference_run),
    "yolov5_commit": repo_commit,
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "data": str(data_yaml),
    "image_counts": counts,
    "command": command,
}
(project_dir / f"{run_name}_environment.json").write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Training results will be saved to: ", run_dir)
print("Full log: ", log_path)

status = display("Starting YOLOv5s training...", display_id=True)

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command,
        cwd=str(repo),
        stdout=log,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )

    try:
        while process.poll() is None:
            time.sleep(10)
            with log_path.open("rb") as reader:
                reader.seek(max(0, log_path.stat().st_size - 6000))
                tail = reader.read().decode("utf-8", errors="replace")
            status.update("\n".join(tail.splitlines()[-12:]))
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise

if process.returncode != 0:
    print(log_path.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise RuntimeError(f"Training failed; see log: {log_path}")

best_pt = run_dir / "weights/best.pt"
assert best_pt.is_file(), "Training exited without producing best.pt"

print("Training complete")
print("Best weights: ", best_pt)

Training results will be saved to:  /content/drive/MyDrive/paper_cup_yolov5s/v5s_416_full120_20261124_122733
Full log:  /content/drive/MyDrive/paper_cup_yolov5s/v5s_416_full120_20261124_122733.log


'\x1b[K                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% ━━━━━━━━━━━━ 2/2 8.9it/s 0.2s\n                   all         26         78      0.919      0.952      0.933      0.754\n\n      Epoch    GPU_mem   box_loss   obj_loss   cls_loss  Instances       Size\n\n\x1b[K     44/119      1.04G    0.01025   0.002435  0.0001277          8        416: 0% ──────────── 1/242 1.9it/s 0.2s<2:06\n\x1b[K     44/119      1.04G    0.01313    0.00395   0.001242         19        416: 1% ──────────── 3/242 6.0it/s 0.3s<39.7s\n\x1b[K     44/119      1.04G    0.01322   0.004068  0.0008743         17        416: 2% ──────────── 5/242 8.4it/s 0.4s<28.2s\n\x1b[K     44/119      1.04G    0.01354   0.003745  0.0007154         10        416: 2% ──────────── 7/242 10.4it/s 0.6s<22.7s\n\x1b[K     44/119      1.04G    0.01382   0.003903   0.001192         21        416: 3% ──────────── 9/242 11.0it/s 0.7s<21.1s\n\x1b[K     44/119      1.04G    0.01388   0.004034   0.001

'Model summary: 68 layers, 7,026,307 parameters, 0 gradients, 15.8 GFLOPs\n\n\x1b[K                 Class     Images  Instances          P          R      mAP50   mAP50-95: 50% ━━━━━━────── 1/2 2.0it/s 0.1s<0.5s\n\x1b[K                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% ━━━━━━━━━━━━ 2/2 4.9it/s 0.4s\n                   all         26         78      0.915      0.899      0.934      0.774\n                  Back         26          8      0.928          1      0.995      0.767\n                 Front         26          8          1      0.564      0.883      0.766\n             left_side         26          9       0.73          1      0.855       0.79\n            right_side         26         10      0.963        0.9      0.931      0.726\n               upright         26         11      0.916      0.993      0.972      0.764\n           upside_down         26         32      0.956      0.938      0.968      0.832\nResults saved to \x1b/con

Training complete
Best weights:  /content/drive/MyDrive/paper_cup_yolov5s/v5s_416_full120_20261124_122733/weights/best.pt


In [6]:
from pathlib import Path
import subprocess
import sys
import re

run_dir = Path(
    "/content/drive/MyDrive/paper_cup_yolov5s/"
    "v5s_416_full120_20261124_122733"
)
best_pt = run_dir / "weights/best.pt"
data_yaml = Path("/content/2026_GreG-4/data_yolov5.yaml")

assert best_pt.is_file(), f"Weights not found: {best_pt}"
assert data_yaml.is_file(), f"Data config not found: {data_yaml}"

command = [
    sys.executable, "-u", "/content/yolov5/val.py",
    "--weights", str(best_pt),
    "--data", str(data_yaml),
    "--task", "val",
    "--imgsz", "416",
    "--batch-size", "8",
    "--device", "0",
    "--workers", "2",
    "--verbose",
    "--project", str(run_dir),
    "--name", "validation_readable",
    "--exist-ok",
]

print("Validating current YOLOv5s best checkpoint, please wait...", flush=True)

result = subprocess.run(
    command,
    cwd="/content/yolov5",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
)

# Strip ANSI color and progress-bar control codes to display clean logs
clean_output = re.sub(
    r"\x1b\[[0-?]*[ -/]*[@-~]", "", result.stdout
)
print(clean_output)

if result.returncode != 0:
    raise RuntimeError("Validation failed; inspect error output above.")

print("\nValidation complete")
print("Run directory: ", run_dir)
print("Best weights: ", best_pt)
print("Validation output directory: ", run_dir / "validation_readable")

Validating current YOLOv5s best checkpoint, please wait...
val: data=/content/2026_GreG-4/data_yolov5.yaml, weights=['/content/drive/MyDrive/paper_cup_yolov5s/v5s_416_full120_20261124_122733/weights/best.pt'], batch_size=8, imgsz=416, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=0, workers=2, single_cls=False, augment=False, verbose=True, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/paper_cup_yolov5s/v5s_416_full120_20261124_122733, name=validation_readable, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-575-g33ae15ef Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)

Fusing layers... 
Model summary: 68 layers, 7,026,307 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/2026_GreG-4/valid/labels.cache... 26 images, 0 backgrounds, 0 corrupt

                 Class     Images  Instances          P          R      mAP50   mAP50-95: 25% ━━━───────── 1/4 1.1s/it 0.3s<3.4s
                 Clas